In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import warnings

# --- Setup ---
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

## Part 1: Load Purchase Data & Engineer Risk Features

In [2]:
print("--- Part 1: Building Risk Features from Purchases ---")

# Try to load purchase data (supplier/quality data)
purchases_df = None
try:
    purchases_df = pd.read_csv('valio_aimo_purchases_junction_2025.csv')
except FileNotFoundError:
    print("WARNING: valio_aimo_purchases_junction_2025.csv not found.")
    print("Please upload the file to the working directory for accurate risk feature computation.")

if purchases_df is not None:
    # 1. Aggregate by the unique PO line to sum multiple receipts
    # Some datasets use 'po_row_number' as the line identifier; if missing, group only by order_number
    group_cols = ['order_number'] + (['po_row_number'] if 'po_row_number' in purchases_df.columns else [])
    po_agg = purchases_df.groupby(group_cols).agg(
        product_code=('product_code', 'first'),
        ordered_qty=('ordered_qty', 'first'),
        unit=('unit', 'first')

    # 2. Create the supplier-side shortage flag
    po_agg['po_shortage'] = po_agg['ordered_qty'].astype(float) > po_agg['total_received_qty'].astype(float)
,
3
,
Calculating historical risk scores...")
    product_risk = po_agg.groupby('product_code')['po_shortage'].mean().rename('product_risk')
    customer_risk = po_agg.groupby('customer_number')['po_shortage'].mean().rename('customer_risk')
    unit_risk = po_agg.groupby('unit')['po_shortage'].mean().rename('unit_risk')

    print(f"Created {len(product_risk)} product risk scores.")
    print(f"Created {len(customer_risk)} customer risk scores.")

    print("
Top 5 Riskiest Products (from PO data):")
    display(product_risk.sort_values(ascending=False).head())
else:
    # create placeholders so the notebook cells below can run interactively
    po_agg = pd.DataFrame()
    product_risk = pd.Series(dtype=float, name='product_risk')
    customer_risk = pd.Series(dtype=float, name='customer_risk')
    unit_risk = pd.Series(dtype=float, name='unit_risk')
: null,
: []
: 
,
: {
: 

: [
2

# WARNING: The code below must be replaced with real sales data ingestion in production
if 'po_agg' in globals() and not po_agg.empty:
    # For quick iteration we reuse PO-aggregated rows as a placeholder sales_df
    sales_df = po_agg.copy()
    }, inplace=True)
407329
408060
407329

print("Merging risk features into sales data...")
# Merge risk features (left join so new products/customers get NaN then handled)
if not sales_df.empty:
    master_df = sales_df.merge(product_risk.reset_index(), on='product_code', how='left')
    master_df = master_df.merge(customer_risk.reset_index(), on='customer_number', how='left')
    master_df = master_df.merge(unit_risk.reset_index(), on='unit', how='left')
else:
    master_df = pd.DataFrame()

# Create time-based features from the sales order date
if not master_df.empty:
    master_df['order_date'] = pd.to_datetime(master_df['order_date'])
    master_df['order_month'] = master_df['order_date'].dt.month
    master_df['order_dayofweek'] = master_df['order_date'].dt.dayofweek

# Handle missing values (e.g., a new product not in PO history) - keep robust treatment
if not master_df.empty:
    # Do NOT blindly fill with 0 for all columns; fill engineered numeric risks with sensible defaults
    for col in ['product_risk','customer_risk','unit_risk']:

    master_df.fillna({'ordered_qty':0}, inplace=True)
print("Master dataset created and ready for modeling.")

SyntaxError: unterminated string literal (detected at line 25) (4203680451.py, line 25)

## Part 3: Feature Selection & Preparation

In [ ]:
print("--- Part 3: Feature Selection & Preparation ---")

# 1. Define Features (X) and Target (y)
if not master_df.empty:
    y = master_df['TARGET_is_shortage']
    X = master_df[[
        'customer_number',
        'ordered_qty',
        'order_dayofweek',
        'customer_risk',
    ]].copy()
else:
    # Empty placeholders to avoid NameError in subsequent cells
    X = pd.DataFrame()
    y = pd.Series(dtype=int)

# 2. Encode Categorical Features (ML models need numbers, not text)
categorical_cols = ['product_code', 'customer_number', 'unit']
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
if not X.empty:
    X_encoded = X.copy()
    X_encoded[categorical_cols] = encoder.fit_transform(X[categorical_cols])
else:
    X_encoded = X.copy()

# 3. Correlation Analysis (optional plot)
if not X_encoded.empty:
    print("Generating correlation matrix (see plot)...")
    corr_df = X_encoded.copy()
    corr_df['TARGET_is_shortage'] = y.values if len(y)>0 else []
    corr_matrix = corr_df.corr()
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
    plt.title('Feature Correlation Matrix')
    plt.savefig('correlation_matrix.png')
    print("Saved 'correlation_matrix.png'")
else:
    print("No data available for correlation matrix.")

## Part 4: Model Training and Validation

In [ ]:
print("--- Part 4: Model Training and Validation ---")

if not X_encoded.empty and len(y)>0:
    # 1. Split data into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(
        test_size=0.3,
        stratify=y if len(y.unique())>1 else None
,
Training set size: {X_train.shape[0]}")
    print(f"Test set size: {X_test.shape[0]}")

    # 2. Train the Model
    print("Training RandomForestClassifier...")
    model = RandomForestClassifier(
        n_estimators=100,
        n_jobs=-1
,
    print("Model training complete.")

    # 3. Validation
    print("Evaluating model on test data...")
    y_pred = model.predict(X_test)
,
,
,
6
,
,
,
,
,
,
Saved 'confusion_matrix.png'")
else:
    print("Not enough data to train the model.")

## Part 5: Generating Buffer Prediction (Risk Scores)

In [ ]:
print("--- Part 5: Generating 'Buffer Prediction' (Risk Scores) ---")

if 'model' in globals() and not X_test.empty:
    probabilities = model.predict_proba(X_test)[:, 1]
    results_df = X_test.copy()
    results_df['actual_shortage'] = y_test.values
    results_df['predicted_shortage_risk'] = probabilities
    print("Example of risk scores for AI Agent:")
    display(results_df[results_df['actual_shortage'] == 1].sort_values(by='predicted_shortage_risk', ascending=False).head())
    print("
--- How your AI agent would use this ---")
    print("new_risk_score = model.predict_proba(new_order_features)[:, 1][0]")
    print("if new_risk_score > 0.5: # 0.5 is your threshold")
    print("    trigger_ai_call(customer_number, product_code)")
else:
    print("No trained model or test set available to generate buffer predictions.")